# 02 – M0 (W2-05), M1 (W2-06), M1-h (W2-07)

Hướng dẫn từng bước: `docs/huong_dan/W2-05.md`, `W2-06.md`, `W2-07.md`.
Ô có chữ **TODO** là phần m tự viết. Lưu ý: chọn siêu tham số chỉ dựa trên tập **dev**; tập **test** chỉ dùng ở bước đánh giá cuối.

In [ ]:
# Ô 0 – cài đặt môi trường (mỗi lần mở notebook đều chạy ô này trước; Colab hỏi quyền truy cập Drive thì bấm cho phép)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # lối tắt tới Drive chung (xem docs/huong_dan/README.md, mục đầu)
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
from collections import Counter
import re
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
import pandas as pd
from src.data import load_scenario
from src.embeddings import tokenize
from src.evaluate import evaluate_and_save, summarize
from src.utils import set_seed
set_seed(42)
data = {s: load_scenario(s) for s in ['S1', 'S2']}
{s: {k: (len(v) if not isinstance(v, dict) else {t: len(x) for t, x in v.items()}) for k, v in d.items()} for s, d in data.items()}

## M0 – baseline dự đoán nhãn đa số (W2-05): t viết sẵn, m chỉ cần chạy

In [ ]:
for s, d in data.items():
    majority = Counter(d['train']['label']).most_common(1)[0][0]
    print(s, 'nhãn đa số của train:', majority)
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], [majority] * len(df), model='M0', scenario=s, test_set=test_name,
                          seed=42, config={'majority_label': majority}, ids=df['id'])

## M1 – TF-IDF + Logistic Regression / Naive Bayes (W2-06)

Premise và hypothesis được vector hóa TF-IDF **riêng** rồi ghép lại, thêm các **đặc trưng so sánh hai câu** do m viết.
Nếu chỉ dùng TF-IDF, mô hình không có thông tin về quan hệ giữa hai câu: t đã thử, macro-F1 trên dev của ViANLI chỉ khoảng 0,2, thấp hơn mức ngẫu nhiên.

In [ ]:
NEG = re.compile(r'\b(không|chưa|chẳng|chả|đâu có)\b', re.IGNORECASE)

def compare_features(df):
    """Mỗi cặp câu -> một vector đặc trưng. Mọi giá trị phải >= 0 (để dùng được với Naive Bayes)."""
    rows = []
    for p, h in zip(df['premise'], df['hypothesis']):
        tp, th = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
        overlap = len(tp & th) / max(1, len(th))          # tỉ lệ từ của hypothesis xuất hiện trong premise
        # TODO a: số từ phủ định trong premise và trong hypothesis: len(NEG.findall(p)), len(NEG.findall(h))
        # TODO b: chênh lệch độ dài hai câu (số âm tiết) chia cho 50:  abs(len(tokenize(p)) - len(tokenize(h))) / 50
        # TODO c: hypothesis có số nào không xuất hiện trong premise? có -> 1.0, không -> 0.0  (dùng re.findall(r'\d+', ...))
        rows.append([overlap])                             # TODO: thêm a, b, c vào danh sách này
    return np.array(rows)

def make_vectorizers():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True))

def features(df, vp, vh, fit=False):
    P = vp.fit_transform(df['premise']) if fit else vp.transform(df['premise'])
    H = vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])
    return sp.hstack([P, H, sp.csr_matrix(compare_features(df))]).tocsr()

In [ ]:
# Chọn siêu tham số C của Logistic Regression dựa trên macro-F1 tập DEV. Xong S2 thì sửa s = 'S1' và chạy lại
s = 'S2'
d = data[s]
vp, vh = make_vectorizers()
X_train = features(d['train'], vp, vh, fit=True)
X_dev = features(d['dev'], vp, vh)
for C in [0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=2000).fit(X_train, d['train']['label'])
    print(f'C={C}  dev macro-F1 = {f1_score(d["dev"]["label"], model.predict(X_dev), average="macro"):.3f}')

**TODO d:** làm tương tự với Naive Bayes `MultinomialNB(alpha=...)`, thử alpha = 0.1, 0.5, 1.0. Ghi bảng macro-F1 trên dev của cả hai bộ phân loại.

**TODO e:** với **từng** kịch bản, chọn bộ phân loại và siêu tham số có macro-F1 dev cao nhất, rồi đánh giá trên tập test:
```python
for test_name, df in d['test'].items():
    evaluate_and_save(df['label'], best_model.predict(features(df, vp, vh)), model='M1', scenario=s,
                      test_set=test_name, seed=42, config={'clf': 'LR', 'C': best_C, 'features': '...', 'dev_macro_f1': best_dev_f1},
                      ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
```

In [ ]:
# TODO d, e

## M1-h – mô hình chỉ dùng hypothesis (W2-07)

**TODO f:** giống M1 nhưng **chỉ** dùng TF-IDF của hypothesis (không dùng premise, không dùng `compare_features`). Chọn C trên dev,
lưu với `model='M1h'`. Nếu M1-h cao hơn hẳn M0 thì hypothesis chứa dấu vết của người gán nhãn (annotation artifacts).
Đoạn code mở đầu có trong `docs/huong_dan/W2-07.md`.

In [ ]:
# TODO f

In [ ]:
summarize().query("model in ['M0', 'M1', 'M1h']")

---
## W3-09 – Phân tích loại bỏ đặc trưng (ablation) cho M1

Hướng dẫn: `docs/huong_dan/W3-09.md`. Cần chạy trước: ô 0, ô tải dữ liệu, ô định nghĩa `compare_features` (phần M1).
Chỉ dùng tập **dev**. Bộ phân loại và siêu tham số giữ nguyên như đã chọn ở W2-06.

In [ ]:
# TODO: điền bộ phân loại + siêu tham số đã chọn ở W2-06 cho từng kịch bản
MO_HINH = {'S1': lambda: LogisticRegression(C=1, max_iter=2000),
           'S2': lambda: LogisticRegression(C=1, max_iter=2000)}

# Nhóm đặc trưng. Số trong danh sách là chỉ số cột của compare_features (xem bảng trong hướng dẫn W3-09)
NHOM = {'tfidf_p': None, 'tfidf_h': None, 'overlap': [0], 'phu_dinh': [1, 2], 'do_dai': [3], 'so': [4]}
SO_SANH = ['overlap', 'phu_dinh', 'do_dai', 'so']

def tao_khoi(d):
    """Tính sẵn ma trận của từng nhóm đặc trưng cho train và dev (chỉ tính 1 lần)."""
    vp, vh = make_vectorizers()
    khoi = {}
    for tap in ['train', 'dev']:
        df, fit = d[tap], tap == 'train'
        khoi[tap] = {'tfidf_p': vp.fit_transform(df['premise']) if fit else vp.transform(df['premise']),
                     'tfidf_h': vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])}
        F = compare_features(df)
        for ten in SO_SANH:
            khoi[tap][ten] = sp.csr_matrix(F[:, NHOM[ten]])
    return khoi

def ghep(khoi_tap, cac_nhom):
    return sp.hstack([khoi_tap[n] for n in cac_nhom]).tocsr()

cau_hinh = [('(đủ đặc trưng)', list(NHOM))]
cau_hinh += [(f'bỏ {n}', [m for m in NHOM if m != n]) for n in NHOM]
cau_hinh += [('chỉ TF-IDF', ['tfidf_p', 'tfidf_h']), ('chỉ đặc trưng so sánh', SO_SANH)]

ket_qua = []
for s in ['S2', 'S1']:
    d = data[s]
    khoi = tao_khoi(d)
    for ten, cac_nhom in cau_hinh:
        model = MO_HINH[s]().fit(ghep(khoi['train'], cac_nhom), d['train']['label'])
        f1 = f1_score(d['dev']['label'], model.predict(ghep(khoi['dev'], cac_nhom)), average='macro')
        ket_qua.append({'kich_ban': s, 'cau_hinh': ten, 'dev_macro_f1': round(f1, 4)})
        print(s, ten, round(f1, 4))

ablation = pd.DataFrame(ket_qua)
ablation['chenh_lech'] = (ablation['dev_macro_f1']
                          - ablation.groupby('kich_ban')['dev_macro_f1'].transform('first')).round(4)
ablation

In [ ]:
# Lưu bảng ablation (vào Drive chung, thư mục results/analysis)
import os
os.makedirs('results/analysis', exist_ok=True)
ablation.to_csv('results/analysis/m1_ablation.csv', index=False, encoding='utf-8-sig')